# Attention Mechanisms

A study refresher. **Attention** is the operation at the heart of the Transformer: every position in a sequence builds its new representation by **looking at every other position** and taking a weighted average, where the weights are computed *dynamically* from the content itself.

**Domain:** Architectures  ·  **runnable:** yes

## 1. What & Why

**What it is.** Attention maps a set of **queries** against a set of **keys** to produce a similarity score for every query–key pair, turns those scores into a probability distribution (softmax), and uses it to take a weighted sum of **values**. The canonical form is **scaled dot-product attention** (Vaswani et al., 2017):

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**The problem it solves.** Before attention, sequence models (RNNs/LSTMs) compressed an entire input into a single fixed-size hidden state, which became an information bottleneck for long inputs and forced strictly sequential computation. Attention lets any output position reach **directly** to any input position in **one step** — the path length between two tokens is O(1) instead of O(n) — and the whole operation is a couple of matrix multiplies, so it **parallelizes** across the sequence on a GPU.

**When to reach for it.** Any time relationships between elements matter and those relationships are *content-dependent* rather than fixed by position: language, code, sets, graphs, image patches, audio frames. Attention is the default building block for modern sequence and set models.

**When NOT to.** When the sequence is very long and you only need local context — vanilla self-attention is **O(n²)** in time and memory, so 100k-token contexts get expensive fast. Pure convolutions, state-space models (Mamba), or sparse/linear-attention variants can be cheaper. See §7.

## 2. Mental Model

**Attention is soft, differentiable dictionary lookup.**

A Python dict does a *hard* lookup: `d[key]` returns the one value whose key matches exactly. Attention does a **soft** lookup — instead of matching one key, the query measures its similarity to *every* key, normalizes those similarities into weights that sum to 1, and returns a **blend of all the values** weighted by how well each key matched.

- **Query (Q):** "what am I looking for?" — emitted by the position doing the looking.
- **Key (K):** "what do I offer?" — an advertisement each position publishes.
- **Value (V):** "what I actually hand over if you attend to me" — the payload.

The dot product `q·k` is the relevance score; `√d_k` keeps it from blowing up; softmax turns scores into a weighted average. Because it's all matrix multiplication and softmax, it's **fully differentiable**, so the model *learns* what to put in Q, K, and V end-to-end.

**Self vs cross.** In **self-attention**, Q, K, V all come from the same sequence (tokens relate to each other). In **cross-attention**, Q comes from one sequence (e.g. the decoder) and K, V from another (e.g. the encoder) — this is how a translator's output attends to the source sentence.

## 3. Key Concepts

- **Query / Key / Value.** Three linear projections of the input. Q and K live in the same space so their dot product is meaningful; V can have a different dimension.
- **Scaled dot-product.** Scores are `QKᵀ / √d_k`. The `√d_k` scaling counteracts dot products growing with dimension — without it, large logits push softmax into saturated regions where gradients vanish.
- **Softmax → weights.** Each query produces a distribution over all keys (rows sum to 1). These weights are interpretable: an "attention map."
- **Multi-head attention (MHA).** Run `h` attention operations in parallel on lower-dimensional slices (`d_k = d_model / h`), then concatenate and project. Each head can specialize (syntax, coreference, position). More heads ≠ strictly better; it's about representational subspaces.
- **Masking.** A boolean/additive mask sets disallowed scores to `−∞` before softmax. **Causal (look-ahead) masks** stop a position from seeing the future (autoregressive decoding); **padding masks** ignore padding tokens in a batch.
- **Positional information.** Attention is **permutation-equivariant** — it has no built-in notion of order. Order is injected separately via positional encodings/embeddings (sinusoidal, learned, or rotary/RoPE).
- **Complexity.** Time and memory are **O(n²·d)** for sequence length n — the defining cost and the thing every efficient-attention variant attacks.
- **FlashAttention.** Not a new math; an **IO-aware** exact-attention kernel that tiles the computation to avoid materializing the n×n matrix in slow HBM, giving big speed/memory wins in practice.

## 4. Setup

The worked examples below need only **NumPy**, so they run anywhere on CPU — no GPU, no downloads. The final cell shows the equivalent in **PyTorch** (`torch.nn.functional.scaled_dot_product_attention`, which dispatches to FlashAttention when available) and is **gated** behind an import check, so the notebook still executes top-to-bottom without PyTorch installed.

```bash
pip install numpy
pip install torch     # optional — only for the last cell
```

In [1]:
import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
print("NumPy", np.__version__, "ready — attention demos run on CPU.")

NumPy 2.5.0 ready — attention demos run on CPU.


## 5. Worked Examples

### Example 1 — Scaled dot-product attention from scratch

We implement the core equation and run it on a tiny toy sequence. We deliberately craft a query that is aligned with one particular key so you can see the softmax concentrate its weight there.

In [2]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)      # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)   # (..., n_q, n_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)     # disallowed -> -inf
    weights = softmax(scores, axis=-1)               # rows sum to 1
    return weights @ V, weights

# Toy sequence: 4 tokens, model dim 8.
n, d = 4, 8
K = rng.standard_normal((n, d))
V = rng.standard_normal((n, d))

# Make one query point straight at token 2's key -> attention should spike there.
q = K[2] * 3.0
out, w = scaled_dot_product_attention(q[None, :], K, V)

print("attention weights over the 4 keys:", w[0])
print("argmax key:", w[0].argmax(), "(we aimed the query at key 2)")
print("output ~= V[2]?", np.allclose(out[0], V[2], atol=0.4))

attention weights over the 4 keys: [0.018 0.04  0.937 0.005]
argmax key: 2 (we aimed the query at key 2)
output ~= V[2]? True


### Example 2 — Multi-head self-attention + a causal mask

Now the full self-attention block: project the input into Q/K/V, split into heads, attend, concat, and project out. We also apply a **causal mask** so each position can only attend to itself and earlier positions — exactly what a GPT-style decoder does. Notice the attention matrix is **lower-triangular**.

In [3]:
def multi_head_self_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=False):
    n, d_model = X.shape
    d_head = d_model // n_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv                 # (n, d_model)

    # (n, d_model) -> (n_heads, n, d_head)
    split = lambda M: M.reshape(n, n_heads, d_head).transpose(1, 0, 2)
    Qh, Kh, Vh = split(Q), split(K), split(V)

    mask = None
    if causal:
        mask = np.tril(np.ones((n, n), dtype=bool))  # allow j <= i
    ctx, weights = scaled_dot_product_attention(Qh, Kh, Vh, mask=mask)

    ctx = ctx.transpose(1, 0, 2).reshape(n, d_model)  # concat heads
    return ctx @ Wo, weights                          # (n, d_model), (h, n, n)

n, d_model, n_heads = 5, 16, 4
X = rng.standard_normal((n, d_model))
Wq, Wk, Wv, Wo = (rng.standard_normal((d_model, d_model)) / np.sqrt(d_model)
                  for _ in range(4))

out, weights = multi_head_self_attention(X, Wq, Wk, Wv, Wo, n_heads, causal=True)
print("output shape:", out.shape, "| per-head attention shape:", weights.shape)
print("\nHead 0 causal attention (lower-triangular, rows sum to 1):")
print(weights[0])
print("\nrow sums:", weights[0].sum(axis=1))

output shape: (5, 16) | per-head attention shape: (4, 5, 5)

Head 0 causal attention (lower-triangular, rows sum to 1):
[[1.    0.    0.    0.    0.   ]
 [0.277 0.723 0.    0.    0.   ]
 [0.563 0.218 0.219 0.    0.   ]
 [0.26  0.081 0.401 0.258 0.   ]
 [0.441 0.15  0.161 0.169 0.08 ]]

row sums: [1. 1. 1. 1. 1.]


### Example 3 (optional) — The same thing in PyTorch, gated

`torch.nn.functional.scaled_dot_product_attention` is the production path: one call, fused kernels, and an `is_causal` flag. This cell only runs if PyTorch is importable — otherwise it prints the call shape and moves on, so the notebook still executes everywhere.

In [4]:
try:
    import torch
    import torch.nn.functional as F

    torch.manual_seed(0)
    # (batch, heads, seq, head_dim)
    B, H, S, Dh = 1, 4, 5, 4
    q = torch.randn(B, H, S, Dh)
    k = torch.randn(B, H, S, Dh)
    v = torch.randn(B, H, S, Dh)

    # Fused, IO-aware (FlashAttention when available); is_causal builds the mask.
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    print("torch", torch.__version__, "-> output shape:", tuple(out.shape))
except ImportError:
    print("PyTorch not installed — skipping. Call shape would be:")
    print("  F.scaled_dot_product_attention(q, k, v, is_causal=True)")
    print("  with q,k,v of shape (batch, heads, seq, head_dim).")

torch 2.12.1 -> output shape: (1, 4, 5, 4)


## 6. Gotchas & Pitfalls

- **Forgetting the `√d_k` scale.** Without it, dot products grow with dimension, softmax saturates, gradients vanish, and training stalls. It's not optional.
- **Mask sign/direction bugs.** Masks are added *before* softmax and must be `−∞` (not 0, not a small number) for blocked positions. Mixing up "True = keep" vs "True = block" silently lets a decoder peek at the future — your loss looks great and your model is useless at generation time.
- **Numerical softmax overflow.** Always subtract the row max before `exp`. A naive `exp(scores)` overflows for large logits.
- **O(n²) memory ambush.** The n×n score matrix, not the model weights, is what blows up your memory on long sequences. Doubling context **quadruples** attention memory. Use FlashAttention / chunking / sparse variants before you reach for a bigger GPU.
- **Attention weights ≠ explanation.** Tempting to read attention maps as "what the model thinks is important." They're suggestive but **not faithful explanations** — many weight configurations yield the same output, and gradients can disagree. Don't ship them as ground-truth interpretability.
- **No positional info by default.** Self-attention is permutation-equivariant; shuffle the tokens and (absent positional encodings) you get the same shuffled outputs. If word order seems ignored, check that positional encodings are actually applied.
- **Heads vs dimension trade-off.** `d_head = d_model / n_heads`. Cranking `n_heads` up shrinks each head's dimension; too-small heads lose expressivity. It's a partition, not free capacity.
- **Padding leaks.** In batched inputs, unmasked pad tokens contribute to the softmax and contaminate real tokens. Always apply the padding mask.

## 7. When to Use vs Alternatives

| Approach | Strength | Weakness | Reach for it when |
|---|---|---|---|
| **Full self-attention** | Global, content-based, O(1) path length, parallel | O(n²) time & memory; no inductive bias for order/locality | Default for sequences/sets up to a few-thousand tokens |
| **FlashAttention** | *Exact* attention, far less memory, much faster | Needs a supported GPU/kernel; same O(n²) compute | Always, when training/serving on modern GPUs |
| **Sparse / local / sliding-window** (Longformer, BigBird) | Sub-quadratic; handles long docs | Approximate; may miss long-range links | Long documents where attention is mostly local |
| **Linear attention** (Performer, linear kernels) | O(n) time & memory | Often a quality hit; tricky to tune | Very long sequences, quality budget to spare |
| **State-space models** (Mamba/S4) | O(n) recurrence, strong long-range, fast inference | Newer tooling; different mental model | Extremely long sequences, streaming/autoregressive |
| **Convolutions** | Cheap, strong locality bias | Fixed receptive field; no global mixing in one layer | Local patterns dominate (audio front-ends, vision stems) |
| **RNN/LSTM** | O(n) memory, natural streaming | Sequential (slow), long-range bottleneck | Tiny/online settings, or as a baseline |

**Rule of thumb:** start with full attention + FlashAttention. Only move to sparse/linear/SSM variants when a profiler shows attention is your actual bottleneck — premature optimization here usually costs accuracy for little gain. Cross-link: [`transformer`](transformer.ipynb), [`mamba-ssm`](mamba-ssm.ipynb), [`mixture-of-experts`](mixture-of-experts.ipynb).

## 8. Resources

- **"Attention Is All You Need"** (Vaswani et al., 2017) — the original Transformer paper that introduced scaled dot-product and multi-head attention: https://arxiv.org/abs/1706.03762
- **The Illustrated Transformer** (Jay Alammar) — the canonical visual walkthrough of Q/K/V and multi-head attention: https://jalammar.github.io/illustrated-transformer/
- **The Annotated Transformer** (Harvard NLP) — line-by-line PyTorch implementation alongside the paper: https://nlp.seas.harvard.edu/annotated-transformer/
- **FlashAttention** (Dao et al., 2022) — IO-aware exact attention; why memory, not FLOPs, is the bottleneck: https://arxiv.org/abs/2205.14135
- **PyTorch `scaled_dot_product_attention` docs** — the production API used in Example 3: https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html